# Netflix Content Clustering

## Objective

The objective of this task is to group Netflix movies and TV shows
into meaningful clusters using unsupervised machine learning techniques.

K-Means clustering will be used to discover patterns in Netflix content
based on numerical and categorical features.

## Workflow

1. Prepare numerical and categorical features.
2. Apply clustering algorithms.
3. Identify content groups.
4. Visualize clusters.
5. Interpret cluster characteristics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

### Description

In this cell, we import the required Python libraries for Netflix
content clustering.

Pandas and NumPy are used for data manipulation and numerical operations.
Matplotlib is used for data visualization.

Scikit-learn provides the preprocessing, clustering, dimensionality
reduction, and evaluation tools required for this task.

The K-Means algorithm will be used to group similar Netflix titles,
while PCA will help visualize high-dimensional data in two dimensions.

In [ ]:
df = pd.read_csv("../Data/netflix_titles.csv")

df.head()

### Description

The Netflix dataset is loaded from the Data folder using pandas.

The first five rows are displayed to verify that the dataset has been
loaded successfully and to understand its structure.

In [ ]:
print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

### Description

This cell examines the dataset dimensions, data types, and missing values.

Understanding the dataset structure is important before applying
preprocessing and clustering algorithms.

Missing values and data types will help determine the appropriate
preprocessing techniques for numerical and categorical features.

In [ ]:
features = [
    "type",
    "release_year",
    "rating",
    "duration",
    "listed_in",
    "country"
]

cluster_df = df[features].copy()

cluster_df.head()

### Description

A selected set of numerical and categorical features is prepared
for clustering.

The release_year feature represents the year in which the title was
released.

The type, rating, duration, listed_in, and country features provide
information about the content format, audience rating, length,
genres, and production country.

These features will be transformed into a numerical representation
before applying K-Means clustering.

In [ ]:
cluster_df.info()

print("\nMissing Values:")
print(cluster_df.isnull().sum())

### Duration Feature Engineering

In this step, the duration feature is converted from string format into a numerical representation. The numeric value is extracted from values such as "90 min" and "1 Season" using regular expressions.

The resulting `duration_value` feature will be used during the clustering process to represent the duration of each Netflix title numerically.


In [ ]:
cluster_df["duration_value"] = (
    cluster_df["duration"]
    .str.extract(r"(\d+)")
    .astype(int)
)

cluster_df.head()

### Separate Movie Duration and TV Show Seasons

In this step, the duration feature is separated into two numerical features based on the content type.

For movies, the duration is represented in minutes. For TV shows, the duration is represented as the number of seasons.

This separation ensures that movie duration and TV show seasons are treated as different measurements during the clustering process.


In [ ]:
cluster_df["movie_duration"] = np.where(
    cluster_df["type"] == "Movie",
    cluster_df["duration_value"],
    0
)

cluster_df["tv_show_seasons"] = np.where(
    cluster_df["type"] == "TV Show",
    cluster_df["duration_value"],
    0
)

cluster_df.head()

### Categorical Feature Encoding

In this step, categorical features are converted into numerical representations using One-Hot Encoding.

The selected categorical features include content type, audience rating, genres, and production country.

One-Hot Encoding creates binary columns for different categories, allowing the K-Means clustering algorithm to process categorical information numerically.

This transformation helps the model identify similarities and differences between Netflix titles based on their categorical characteristics.


In [ ]:
categorical_features = [
    "type",
    "rating",
    "listed_in",
    "country"
]

numerical_features = [
    "release_year",
    "movie_duration",
    "tv_show_seasons"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

X = preprocessor.fit_transform(cluster_df)

print("Feature Matrix Shape:", X.shape)

### Determining the Optimal Number of Clusters

In this step, the Elbow Method is used to determine a suitable number of clusters for the K-Means algorithm.

K-Means is tested with different values of K, and the Within-Cluster Sum of Squares (WCSS), also known as inertia, is recorded for each value.

The value of K where the decrease in inertia begins to slow down can indicate a suitable number of clusters for the dataset.


In [ ]:
inertia = []

k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

print("Inertia values:")
for k, value in zip(k_range, inertia):
    print(f"K={k}: {value:.2f}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(k_range, inertia, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method for Optimal K")

plt.xticks(k_range)
plt.grid(True)

plt.show()

### Silhouette Score Evaluation

The Silhouette Score is used to evaluate the quality of the clusters produced by K-Means.

It measures how similar each data point is to its own cluster compared with other clusters. Higher values generally indicate better-defined separation between clusters.

Silhouette scores will be calculated for different values of K and compared with the Elbow Method results to select a suitable clustering configuration.


In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    cluster_labels = kmeans.fit_predict(X)
    
    score = silhouette_score(X, cluster_labels)
    silhouette_scores.append(score)

print("Silhouette Scores:")

for k, score in zip(range(2, 11), silhouette_scores):
    print(f"K={k}: {score:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(range(2, 11))
plt.grid(True)

plt.show()

### Applying K-Means Clustering

Based on the Silhouette Score evaluation, two clusters are selected for the final K-Means model.

The K-Means algorithm is trained on the preprocessed feature matrix. Each Netflix title is then assigned a cluster label representing the content group to which it is most similar.


In [ ]:
optimal_k = 2

kmeans_final = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_final.fit_predict(X)

cluster_df["cluster"] = cluster_labels

print("Cluster Distribution:")
print(cluster_df["cluster"].value_counts().sort_index())

### Cluster Distribution Analysis

After applying K-Means clustering, each Netflix title was assigned to one of two clusters.

The distribution of titles across the clusters is examined to understand the size of each discovered content group. This provides an initial view of how the unsupervised algorithm segmented the Netflix catalog.


In [ ]:
cluster_summary = cluster_df.groupby("cluster")[
    ["release_year", "movie_duration", "tv_show_seasons"]
].mean()

cluster_summary

In [ ]:
type_distribution = pd.crosstab(
    cluster_df["cluster"],
    cluster_df["type"]
)

type_distribution

### Cluster Analysis by Audience Rating

In this step, the distribution of audience ratings across the discovered clusters is examined.

Comparing rating categories helps identify whether the clusters differ in terms of the intended audience classification of Netflix content.


In [ ]:
rating_distribution = pd.crosstab(
    cluster_df["cluster"],
    cluster_df["rating"]
)

rating_distribution

### Genre Data Preparation

The genre information is expanded so that each genre is represented as a separate observation. The index is reset after exploding the genre column to ensure that the resulting DataFrame has unique index values.

This preparation allows the frequency of genres to be compared across the discovered clusters.


In [ ]:
genre_cluster = cluster_df[["cluster", "listed_in"]].copy()

genre_cluster["listed_in"] = genre_cluster["listed_in"].str.split(", ")

genre_cluster = genre_cluster.explode("listed_in").reset_index(drop=True)

genre_distribution = pd.crosstab(
    genre_cluster["listed_in"],
    genre_cluster["cluster"]
)

genre_distribution

In [ ]:
print("Top Genres in Cluster 0:")
print(
    genre_distribution[0]
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop Genres in Cluster 1:")
print(
    genre_distribution[1]
    .sort_values(ascending=False)
    .head(10)
)

### PCA-Based Cluster Visualization

In this step, Principal Component Analysis (PCA) is used to reduce the high-dimensional feature matrix into two principal components.

The original feature matrix contains 618 dimensions, which cannot be directly visualized in a two-dimensional plot. PCA transforms these features into two components while preserving as much of the important variation in the data as possible.

The resulting two-dimensional representation is then used to visualize the K-Means clusters.


In [ ]:
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X)

print("Original Feature Dimensions:", X.shape)
print("Reduced Feature Dimensions:", X_pca.shape)
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Explained Variance:", pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_labels,
    alpha=0.6
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Netflix Content Clusters using PCA")

plt.show()

### Visualizing Netflix Content Clusters

In this step, the two principal components obtained from PCA are used to visualize the Netflix content clusters in a two-dimensional space.

Each point represents a Netflix title, while the color indicates the K-Means cluster assigned to that title. This visualization helps identify the separation and distribution of the discovered content groups.


In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_labels,
    alpha=0.6
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Netflix Content Clusters using PCA")

plt.show()

### Cluster Interpretation

In this step, the characteristics of the identified Netflix content clusters are interpreted using their size, content type, audience ratings, and genre distributions.

The analysis helps identify the main characteristics of each cluster and provides a meaningful interpretation of the content groups discovered through unsupervised learning.


In [ ]:
cluster_counts = cluster_df["cluster"].value_counts().sort_index()

cluster_percentages = (
    cluster_counts / len(cluster_df) * 100
)

cluster_overview = pd.DataFrame({
    "Number of Titles": cluster_counts,
    "Percentage": cluster_percentages.round(2)
})

cluster_overview

### Interpretation of Cluster Characteristics

The clustering analysis identified two major content groups within the Netflix dataset.

Cluster 0 contains 6,129 titles, representing approximately 69.73% of the dataset. The cluster is overwhelmingly composed of movies, with 6,126 movies and only 3 TV shows. Its most frequent genres include International Movies, Dramas, Comedies, Documentaries, and Action & Adventure.

Cluster 1 contains 2,661 titles, representing approximately 30.27% of the dataset. All titles in this cluster are TV Shows. The most frequent genres include International TV Shows, TV Dramas, TV Comedies, Crime TV Shows, and Kids' TV.

The rating distribution also reflects differences between the two groups, with Cluster 0 containing a broader mixture of movie and general audience ratings, while Cluster 1 contains ratings commonly associated with television content.

Overall, the clustering process discovered a strong separation between movie-oriented and TV-show-oriented content. This separation is expected because the content type was included as one of the clustering features. Therefore, the identified clusters should be interpreted primarily as content-type-based groups rather than completely independent genre-based segments.


## Conclusion

In this task, an unsupervised machine learning approach was used to group Netflix titles into meaningful content clusters.

The dataset was prepared by selecting numerical and categorical features, engineering duration-related features, encoding categorical variables using One-Hot Encoding, and scaling numerical features. K-Means clustering was then applied to the transformed feature matrix.

The Elbow Method and Silhouette Score were used to evaluate different cluster configurations. Among the tested values of K, K=2 produced the highest Silhouette Score of 0.3074 and was selected for the final clustering analysis.

The final model divided 8,790 Netflix titles into two clusters. Cluster 0 contained 6,129 titles (69.73%), while Cluster 1 contained 2,661 titles (30.27%). The analysis showed that Cluster 0 was predominantly composed of movies, whereas Cluster 1 consisted entirely of TV shows.

PCA was also used to reduce the 618-dimensional feature space to two principal components for visualization. The first two components explained approximately 50.10% of the total variance, providing a two-dimensional view of the discovered clusters.

This task demonstrated practical applications of K-Means Clustering, data preprocessing, feature engineering, data scaling, cluster evaluation, PCA visualization, and unsupervised machine learning for discovering patterns in Netflix content.
